In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
movies_df = pd.read_csv("/Users/felipepesantez/Documents/development/datasets/recommend_systems/archive/movies.csv")
ratings_df = pd.read_csv("/Users/felipepesantez/Documents/development/datasets/recommend_systems/archive/ratings.csv")

In [19]:
movies_df.drop("genres", inplace=True, axis=1)

In [20]:
df = pd.merge(movies_df, ratings_df, on='movieId')
df.head()

,movieId,title,userId,rating,timestamp
0,1,Toy Story (1995),1,4.0,964982703
1,1,Toy Story (1995),5,4.0,847434962
2,1,Toy Story (1995),7,4.5,1106635946
3,1,Toy Story (1995),15,2.5,1510577970
4,1,Toy Story (1995),17,4.5,1305696483


In [21]:
#CF

In [22]:
#memory based and model based

In [23]:
#SVD

In [24]:
df = df[['userId', 'movieId', 'rating', 'timestamp', 'title']]
df.head()

,userId,movieId,rating,timestamp,title
0,1,1,4.0,964982703,Toy Story (1995)
1,5,1,4.0,847434962,Toy Story (1995)
2,7,1,4.5,1106635946,Toy Story (1995)
3,15,1,2.5,1510577970,Toy Story (1995)
4,17,1,4.5,1305696483,Toy Story (1995)


In [25]:
num_users = df.userId.nunique()
num_items = df.movieId.nunique()
print(num_users, num_items)

610 9724


In [26]:
from sklearn.model_selection import train_test_split

In [27]:
train_data, test_data = train_test_split(df, test_size=0.2)

In [28]:
#memory based

In [29]:
max_user_index = df['userId'].max()
max_item_index = df['movieId'].max()
print(max_user_index, max_item_index)

610 193609


In [30]:
num_users = max_user_index
num_items = max_item_index

In [31]:
user_train_data_matrix = np.zeros((num_users, num_items))

for entry in train_data.itertuples():
    user_index = entry[1] - 1
    item_index = entry[2] - 1
    rating = entry[3]

    user_train_data_matrix[user_index, item_index] = rating

In [32]:
user_test_data_matrix = np.zeros((num_users, num_items))

for entry in test_data.itertuples():
    user_index = entry[1] - 1
    item_index = entry[2] - 1
    rating = entry[3]

    user_test_data_matrix[user_index, item_index] = rating

In [33]:
from sklearn.metrics.pairwise import pairwise_distances

In [18]:
#user_sim = pairwise_distances(user_train_data_matrix, metric='cosine')
#item_sim = pairwise_distances(user_train_data_matrix.T, metric='cosine')

In [19]:
def incremental_pairwise_distances(X, chunk_size=1000, metric='cosine'):
    n_samples = X.shape[0]
    distance_matrix = np.zeros((n_samples, n_samples))

    for i in range(0, n_samples, chunk_size):
        end_i = min(i + chunk_size, n_samples)
        for j in range(0, n_samples, chunk_size):
            end_j = min(j + chunk_size, n_samples)
            distance_matrix[i:end_i, j:end_j] = pairwise_distances(X[i:end_i], X[j:end_j], metric=metric)
    return distance_matrix

In [ ]:
#user_sim = incremental_pairwise_distances(user_train_data_matrix)
#item_sim = incremental_pairwise_distances(user_train_data_matrix.T)

In [34]:
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_distances

svd = TruncatedSVD(n_components=100)
user_train_data_reduced = svd.fit_transform(user_train_data_matrix)

user_sim = cosine_distances(user_train_data_reduced)
item_sim = cosine_distances(user_train_data_reduced.T)

In [35]:
def predict(ratings, sim, type='user'):
    if type == 'user':
        mur = ratings.mean(axis=1)
        r_diff = (ratings - mur[:, np.newaxis])
        pred = mur[:, np.newaxis] + sim.dot(r_diff) / np.array([np.abs(sim).sum(axis=1)]).T
    elif type == 'item':
        pred = ratings.dot(sim) / np.array([np.abs(sim).sum(axis=1)])
    return pred

In [37]:
item_pred = predict(user_train_data_reduced, item_sim, type='item')
user_pred = predict(user_train_data_reduced, user_sim, type='user')

In [38]:
from sklearn.metrics import mean_squared_error

In [39]:
def rmse(pred, ground_truth):
    pred = pred[ground_truth.nonzero()].flatten()
    ground_truth = ground_truth[ground_truth.nonzero()].flatten()
    return np.sqrt(mean_squared_error(pred, ground_truth))

In [40]:
svd_test = TruncatedSVD(n_components=100)
user_test_data_reduced = svd_test.fit_transform(user_test_data_matrix)

In [41]:
print("User based: ", rmse(user_pred, user_test_data_reduced))

User based:  1.9108546765918064


In [42]:
print("Item based: ", rmse(item_pred, user_test_data_reduced))

Item based:  1.7126356859781164
